# To Review the Result For Research Question 1 and 2
### For table 1,  Please Click [Tables 1](#table-rq1)
### For table 2,  Please Click [Tables 2](#table-rq2-1)

In [1]:
# ============================================================
# RQ1 + Dataset Summary Reproduction
# Cell 1: Imports and fixed paths
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import json
import re

import numpy as np
import pandas as pd

from datasets import Dataset, load_dataset


# ============================================================
# Project root
# ============================================================

PROJECT_ROOT = Path(
    "/home/chufeng/Desktop/CSI/ICSE2027"
)


# ============================================================
# NVIDIA raw Open-SWE-Traces
# ============================================================

NVIDIA_ROOT = (
    PROJECT_ROOT
    / "Data Preparation"
    / "dataset"
    / "open_swe_traces"
)


NVIDIA_GROUPS = {
    "OpenHands + MiniMax-M2.5":
        NVIDIA_ROOT
        / "openhands"
        / "minimax_m25",

    "OpenHands + Qwen3.5-122B":
        NVIDIA_ROOT
        / "openhands"
        / "qwen35_122b",

    "SWE-agent + MiniMax-M2.5":
        NVIDIA_ROOT
        / "sweagent"
        / "minimax_m25",

    "SWE-agent + Qwen3.5-122B":
        NVIDIA_ROOT
        / "sweagent"
        / "qwen35_122b",
}


NVIDIA_CONFIG_INFO = {
    "OpenHands + MiniMax-M2.5": {
        "agent": "OpenHands",
        "model": "MiniMax-M2.5",
    },

    "OpenHands + Qwen3.5-122B": {
        "agent": "OpenHands",
        "model": "Qwen3.5-122B",
    },

    "SWE-agent + MiniMax-M2.5": {
        "agent": "SWE",
        "model": "MiniMax-M2.5",
    },

    "SWE-agent + Qwen3.5-122B": {
        "agent": "SWE",
        "model": "Qwen3.5-122B",
    },
}


# ============================================================
# TP annotation root
# ============================================================

TP_ROOT = (
    PROJECT_ROOT
    / "Data Preparation"
    / "data"
)

# NVIDIA annotations from all four configurations are combined
# into one canonical JSONL file.
NVIDIA_TP_FILE = (
    TP_ROOT
    / "nvidia_final.jsonl"
)


# ============================================================
# Nebius
# ============================================================

NEBIUS_DATASET = (
    "nebius/SWE-agent-trajectories"
)

NEBIUS_ANNOTATION_FILE = (
    TP_ROOT
    / "nebius-swe_final.jsonl"
)


NEBIUS_MODELS = [
    "Llama-405B",
    "Llama-70B",
    "Llama-8B",
]


NEBIUS_RAW_MODEL_MAP = {
    "swe-agent-llama-405b":
        "Llama-405B",

    "swe-agent-llama-70b":
        "Llama-70B",

    "swe-agent-llama-8b":
        "Llama-8B",
}


# Annotation files may use either naming style
NEBIUS_ANNOTATION_MODEL_MAP = {
    "swe-agent-llama-405b":
        "Llama-405B",

    "swe-agent-llama-70b":
        "Llama-70B",

    "swe-agent-llama-8b":
        "Llama-8B",

    "llama-405b":
        "Llama-405B",

    "llama-70b":
        "Llama-70B",

    "llama-8b":
        "Llama-8B",
}


# ============================================================
# Output
# ============================================================

RQ1_OUTPUT_DIR = (
    TP_ROOT
    / "rq1_final"
)

RQ1_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    f"Output directory:\n"
    f"{RQ1_OUTPUT_DIR}"
)


/home/chufeng/Desktop/CSI/ICSE2027/.venv/lib/python3.13/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output directory:
/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/rq1_final


In [2]:
# ============================================================
# Cell 2: Path sanity checks
# ============================================================

print("=" * 100)
print("NVIDIA RAW DATA")
print("=" * 100)

for config, folder in NVIDIA_GROUPS.items():

    arrow_files = (
        sorted(
            folder.rglob("*.arrow")
        )
        if folder.exists()
        else []
    )

    print(
        f"{config:35s} "
        f"exists={folder.exists()} "
        f"arrow_files={len(arrow_files)}"
    )


print("\n" + "=" * 100)
print("NVIDIA TP ANNOTATION")
print("=" * 100)

print(
    f"exists={NVIDIA_TP_FILE.exists()} "
    f"{NVIDIA_TP_FILE.name}"
)


print("\n" + "=" * 100)
print("NEBIUS ANNOTATION")
print("=" * 100)

print(
    "Path   :",
    NEBIUS_ANNOTATION_FILE
)

print(
    "Exists :",
    NEBIUS_ANNOTATION_FILE.exists()
)


assert all(
    folder.exists()
    for folder in NVIDIA_GROUPS.values()
)

assert NVIDIA_TP_FILE.exists()

assert (
    NEBIUS_ANNOTATION_FILE.exists()
)

NVIDIA RAW DATA
OpenHands + MiniMax-M2.5            exists=True arrow_files=29
OpenHands + Qwen3.5-122B            exists=True arrow_files=36
SWE-agent + MiniMax-M2.5            exists=True arrow_files=32
SWE-agent + Qwen3.5-122B            exists=True arrow_files=32

NVIDIA TP ANNOTATION
exists=True nvidia_final.jsonl

NEBIUS ANNOTATION
Path   : /home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/nebius-swe_final.jsonl
Exists : True


In [3]:
# ============================================================
# Cell 3: Raw-stat helper
# ============================================================

def new_raw_stats():

    return {
        # trajectory-level
        "total_trajectories":
            0,

        "successful_trajectories":
            0,

        "non_successful_trajectories":
            0,

        # PR-level
        "all_prs":
            set(),

        "successful_attempt_prs":
            set(),

        "non_successful_attempt_prs":
            set(),

        # trajectory IDs
        # NVIDIA uses these for annotation alignment
        "all_trajectory_ids":
            set(),

        "successful_trajectory_ids":
            set(),

        "non_successful_trajectory_ids":
            set(),
    }


In [4]:
# ============================================================
# Cell 4: Scan NVIDIA raw Arrow trajectories
# ============================================================

nvidia_stats = {}


for config, folder in NVIDIA_GROUPS.items():

    print("\n" + "=" * 100)
    print(config)
    print("=" * 100)

    s = new_raw_stats()

    arrow_files = sorted(
        folder.rglob("*.arrow")
    )

    if not arrow_files:

        raise FileNotFoundError(
            f"No Arrow files found:\n{folder}"
        )


    for arrow_file in arrow_files:

        ds = Dataset.from_file(
            str(arrow_file)
        )


        required = {
            "instance_id",
            "trajectory_id",
            "resolved",
        }

        missing = (
            required
            - set(ds.column_names)
        )

        if missing:

            raise ValueError(
                f"{arrow_file} "
                f"missing columns: {missing}"
            )


        for instance_id, trajectory_id, resolved in zip(
            ds["instance_id"],
            ds["trajectory_id"],
            ds["resolved"],
        ):

            instance_id = str(
                instance_id
            ).strip()

            trajectory_id = str(
                trajectory_id
            ).strip()

            resolved = int(
                resolved
            )


            # ----------------------------------------------
            # Overall coverage
            # ----------------------------------------------

            s[
                "total_trajectories"
            ] += 1

            s[
                "all_prs"
            ].add(
                instance_id
            )

            s[
                "all_trajectory_ids"
            ].add(
                trajectory_id
            )


            # ----------------------------------------------
            # Successful
            # ----------------------------------------------

            if resolved == 1:

                s[
                    "successful_trajectories"
                ] += 1

                s[
                    "successful_attempt_prs"
                ].add(
                    instance_id
                )

                s[
                    "successful_trajectory_ids"
                ].add(
                    trajectory_id
                )


            # ----------------------------------------------
            # Non-successful
            # ----------------------------------------------

            elif resolved in {
                0,
                -1,
            }:

                s[
                    "non_successful_trajectories"
                ] += 1

                s[
                    "non_successful_attempt_prs"
                ].add(
                    instance_id
                )

                s[
                    "non_successful_trajectory_ids"
                ].add(
                    trajectory_id
                )


            else:

                raise ValueError(
                    f"Unexpected resolved value: "
                    f"{resolved}"
                )


    nvidia_stats[
        config
    ] = s


    print(
        f"Trajectories       : "
        f"{s['total_trajectories']:,}"
    )

    print(
        f"Non-successful traj: "
        f"{s['non_successful_trajectories']:,}"
    )

    print(
        f"Unique PRs         : "
        f"{len(s['all_prs']):,}"
    )

    print(
        f"Successful PRs     : "
        f"{len(s['successful_attempt_prs']):,}"
    )

    print(
        f"Non-successful PRs : "
        f"{len(s['non_successful_attempt_prs']):,}"
    )



OpenHands + MiniMax-M2.5
Trajectories       : 49,948
Non-successful traj: 34,007
Unique PRs         : 20,098
Successful PRs     : 6,993
Non-successful PRs : 14,885

OpenHands + Qwen3.5-122B
Trajectories       : 55,488
Non-successful traj: 41,831
Unique PRs         : 20,362
Successful PRs     : 6,183
Non-successful PRs : 16,752

SWE-agent + MiniMax-M2.5
Trajectories       : 57,268
Non-successful traj: 37,717
Unique PRs         : 20,791
Successful PRs     : 8,106
Non-successful PRs : 15,164

SWE-agent + Qwen3.5-122B
Trajectories       : 44,785
Non-successful traj: 28,690
Unique PRs         : 18,211
Successful PRs     : 7,350
Non-successful PRs : 12,977


In [5]:
# ============================================================
# Cell 5: NVIDIA raw dataset-summary table
# ============================================================

nvidia_dataset_rows = []


for config in NVIDIA_GROUPS:

    s = nvidia_stats[
        config
    ]

    info = NVIDIA_CONFIG_INFO[
        config
    ]


    nvidia_dataset_rows.append(
        {
            "Dataset":
                "NVIDIA",

            "Agent":
                info["agent"],

            "Model":
                info["model"],

            "Traj.":
                s[
                    "total_trajectories"
                ],

            "Non-Successful Traj.":
                s[
                    "non_successful_trajectories"
                ],

            "Unique PRs":
                len(
                    s[
                        "all_prs"
                    ]
                ),

            "Successful Attempt PRs":
                len(
                    s[
                        "successful_attempt_prs"
                    ]
                ),

            "Non-Successful Attempt PRs":
                len(
                    s[
                        "non_successful_attempt_prs"
                    ]
                ),
        }
    )


nvidia_dataset_summary = pd.DataFrame(
    nvidia_dataset_rows
)


display(
    nvidia_dataset_summary.style.format(
        {
            "Traj.":
                "{:,}",

            "Non-Successful Traj.":
                "{:,}",

            "Unique PRs":
                "{:,}",

            "Successful Attempt PRs":
                "{:,}",

            "Non-Successful Attempt PRs":
                "{:,}",
        }
    )
)


,Dataset,Agent,Model,Traj.,Non-Successful Traj.,Unique PRs,Successful Attempt PRs,Non-Successful Attempt PRs
0,NVIDIA,OpenHands,MiniMax-M2.5,"49,948","34,007","20,098","6,993","14,885"
1,NVIDIA,OpenHands,Qwen3.5-122B,"55,488","41,831","20,362","6,183","16,752"
2,NVIDIA,SWE,MiniMax-M2.5,"57,268","37,717","20,791","8,106","15,164"
3,NVIDIA,SWE,Qwen3.5-122B,"44,785","28,690","18,211","7,350","12,977"


In [6]:
# ============================================================
# Cell 6: Load combined canonical NVIDIA TP annotations
# ============================================================

# The four NVIDIA annotation files have been merged into:
#     TP_ROOT / "nvidia_final.jsonl"
#
# To keep every downstream analysis unchanged, this cell reconstructs
# the same per-configuration structure:
#     nvidia_tp[config]["tp_trajectories"]
#     nvidia_tp[config]["tp_prs"]
#
# Configuration assignment is recovered from trajectory_id by matching
# against the raw NVIDIA trajectory IDs scanned in Cell 4. This does not
# require agent/model metadata to be present in the combined JSONL file.


def infer_nvidia_config_from_metadata(record):
    """Best-effort fallback if trajectory_id is not uniquely identifying."""

    candidate_keys = [
        "config",
        "configuration",
        "agent",
        "agent_name",
        "framework",
        "model",
        "model_name",
        "source",
        "source_file",
        "file",
        "filename",
    ]

    metadata_text = " ".join(
        str(record.get(key, ""))
        for key in candidate_keys
    ).lower()

    if "openhands" in metadata_text:
        agent = "OpenHands"
    elif (
        "swe-agent" in metadata_text
        or "sweagent" in metadata_text
        or "swe_agent" in metadata_text
    ):
        agent = "SWE-agent"
    else:
        agent = None

    if "minimax" in metadata_text:
        model = "MiniMax-M2.5"
    elif "qwen" in metadata_text:
        model = "Qwen3.5-122B"
    else:
        model = None

    if agent is None or model is None:
        return None

    return f"{agent} + {model}"


# ------------------------------------------------------------
# 1. Read the combined file once and split rows by configuration
# ------------------------------------------------------------

canonical_by_config = {
    config: {}
    for config in NVIDIA_GROUPS
}

raw_rows_by_config = Counter()
duplicate_rows_by_config = Counter()

combined_raw_rows = 0
skipped_missing_trajectory_id = 0


with NVIDIA_TP_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        trajectory_id = record.get(
            "trajectory_id"
        )

        if trajectory_id is None:
            skipped_missing_trajectory_id += 1
            continue

        trajectory_id = str(
            trajectory_id
        ).strip()

        if not trajectory_id:
            skipped_missing_trajectory_id += 1
            continue

        combined_raw_rows += 1

        # Primary routing: trajectory_id membership in raw NVIDIA data.
        matching_configs = [
            config
            for config in NVIDIA_GROUPS
            if trajectory_id in nvidia_stats[config]["all_trajectory_ids"]
        ]

        # Fallback: use any agent/model provenance retained in the merged file.
        if len(matching_configs) != 1:
            metadata_config = infer_nvidia_config_from_metadata(
                record
            )

            if (
                metadata_config in NVIDIA_GROUPS
                and trajectory_id
                in nvidia_stats[metadata_config]["all_trajectory_ids"]
            ):
                matching_configs = [metadata_config]

        if len(matching_configs) != 1:
            raise ValueError(
                "Cannot uniquely map NVIDIA annotation to one raw "
                "configuration:\n"
                f"  line={line_number}\n"
                f"  trajectory_id={trajectory_id}\n"
                f"  matching_configs={matching_configs}\n"
                f"  metadata_fallback="
                f"{infer_nvidia_config_from_metadata(record)}"
            )

        config = matching_configs[0]

        raw_rows_by_config[config] += 1

        if trajectory_id in canonical_by_config[config]:
            duplicate_rows_by_config[config] += 1

        # Preserve the original policy: last occurrence wins.
        canonical_by_config[config][trajectory_id] = record


print("=" * 100)
print("NVIDIA COMBINED ANNOTATION ROUTING")
print("=" * 100)
print(f"Input file                 : {NVIDIA_TP_FILE}")
print(f"Rows with trajectory_id    : {combined_raw_rows:,}")
print(
    "Rows missing trajectory_id: "
    f"{skipped_missing_trajectory_id:,}"
)


# ------------------------------------------------------------
# 2. Reconstruct the original nvidia_tp[config] interface
# ------------------------------------------------------------

# Keeping this interface unchanged means all later RQ1/RQ2 cells
# continue to work without modification.
nvidia_tp = {}


for config in NVIDIA_GROUPS:

    canonical = canonical_by_config[config]

    tp_trajectory_ids = set()
    tp_prs = set()

    for trajectory_id, record in canonical.items():

        classification = str(
            record.get(
                "classification",
                "",
            )
        ).strip().upper()

        if classification != "TEST_PANDERING":
            continue

        instance_id = record.get(
            "instance_id"
        )

        if instance_id is None:
            raise ValueError(
                f"Missing instance_id: "
                f"{config} / {trajectory_id}"
            )

        instance_id = str(
            instance_id
        ).strip()

        tp_trajectory_ids.add(
            trajectory_id
        )

        tp_prs.add(
            instance_id
        )

    # --------------------------------------------------------
    # TP must be inside the non-successful study population
    # --------------------------------------------------------

    outside_trajectory_population = (
        tp_trajectory_ids
        - nvidia_stats[config][
            "non_successful_trajectory_ids"
        ]
    )

    outside_pr_population = (
        tp_prs
        - nvidia_stats[config][
            "non_successful_attempt_prs"
        ]
    )

    assert len(outside_trajectory_population) == 0, (
        f"{config}: TP trajectories outside non-successful "
        f"population: {len(outside_trajectory_population):,}"
    )

    assert len(outside_pr_population) == 0, (
        f"{config}: TP PRs outside non-successful "
        f"population: {len(outside_pr_population):,}"
    )

    nvidia_tp[config] = {
        "raw_annotation_rows":
            raw_rows_by_config[config],

        "duplicate_rows":
            duplicate_rows_by_config[config],

        "canonical_annotation_rows":
            len(canonical),

        "tp_trajectories":
            tp_trajectory_ids,

        "tp_prs":
            tp_prs,
    }

    print("\n" + config)

    print(
        f"Raw annotation rows : "
        f"{raw_rows_by_config[config]:,}"
    )

    print(
        f"Duplicate rows      : "
        f"{duplicate_rows_by_config[config]:,}"
    )

    print(
        f"Canonical rows      : "
        f"{len(canonical):,}"
    )

    print(
        f"TP trajectories     : "
        f"{len(tp_trajectory_ids):,}"
    )

    print(
        f"TP-affected PRs     : "
        f"{len(tp_prs):,}"
    )


assert (
    sum(raw_rows_by_config.values())
    == combined_raw_rows
), (
    "Not every combined NVIDIA annotation row was assigned "
    "to exactly one configuration."
)

print("\n✓ Combined NVIDIA annotations were routed successfully.")


NVIDIA COMBINED ANNOTATION ROUTING
Input file                 : /home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/nvidia_final.jsonl
Rows with trajectory_id    : 3,464
Rows missing trajectory_id: 0

OpenHands + MiniMax-M2.5
Raw annotation rows : 1,059
Duplicate rows      : 0
Canonical rows      : 1,059
TP trajectories     : 1,059
TP-affected PRs     : 950

OpenHands + Qwen3.5-122B
Raw annotation rows : 759
Duplicate rows      : 0
Canonical rows      : 759
TP trajectories     : 759
TP-affected PRs     : 707

SWE-agent + MiniMax-M2.5
Raw annotation rows : 936
Duplicate rows      : 0
Canonical rows      : 936
TP trajectories     : 936
TP-affected PRs     : 819

SWE-agent + Qwen3.5-122B
Raw annotation rows : 710
Duplicate rows      : 0
Canonical rows      : 710
TP trajectories     : 710
TP-affected PRs     : 661

✓ Combined NVIDIA annotations were routed successfully.


In [7]:
# ============================================================
# Cell 7: Scan raw Nebius dataset
# ============================================================

nebius_stats = {
    model:
        new_raw_stats()

    for model in NEBIUS_MODELS
}


unknown_models = Counter()


dataset = load_dataset(
    NEBIUS_DATASET,
    split="train",
    streaming=True,
)


for row in dataset:

    instance_id = str(
        row[
            "instance_id"
        ]
    ).strip()

    raw_model = str(
        row[
            "model_name"
        ]
    ).strip().lower()

    target = row[
        "target"
    ]


    model = (
        NEBIUS_RAW_MODEL_MAP.get(
            raw_model
        )
    )


    if model is None:

        unknown_models[
            raw_model
        ] += 1

        continue


    s = nebius_stats[
        model
    ]


    # --------------------------------------------------------
    # Overall
    # --------------------------------------------------------

    s[
        "total_trajectories"
    ] += 1

    s[
        "all_prs"
    ].add(
        instance_id
    )


    # --------------------------------------------------------
    # Successful
    # --------------------------------------------------------

    if target is True:

        s[
            "successful_trajectories"
        ] += 1

        s[
            "successful_attempt_prs"
        ].add(
            instance_id
        )


    # --------------------------------------------------------
    # Non-successful
    # --------------------------------------------------------

    elif target is False:

        s[
            "non_successful_trajectories"
        ] += 1

        s[
            "non_successful_attempt_prs"
        ].add(
            instance_id
        )


    else:

        raise ValueError(
            f"Unexpected Nebius target: "
            f"{target!r}"
        )


print("=" * 100)
print("NEBIUS MODEL CHECK")
print("=" * 100)

print(
    "Unknown models:",
    dict(
        unknown_models
    )
)


NEBIUS MODEL CHECK
Unknown models: {}


In [8]:
# ============================================================
# Cell 8: Nebius raw dataset summary
# ============================================================

nebius_dataset_rows = []


for model in NEBIUS_MODELS:

    s = nebius_stats[
        model
    ]


    nebius_dataset_rows.append(
        {
            "Dataset":
                "Nebius",

            "Agent":
                "SWE",

            "Model":
                model,

            "Traj.":
                s[
                    "total_trajectories"
                ],

            "Non-Successful Traj.":
                s[
                    "non_successful_trajectories"
                ],

            "Unique PRs":
                len(
                    s[
                        "all_prs"
                    ]
                ),

            "Successful Attempt PRs":
                len(
                    s[
                        "successful_attempt_prs"
                    ]
                ),

            "Non-Successful Attempt PRs":
                len(
                    s[
                        "non_successful_attempt_prs"
                    ]
                ),
        }
    )


nebius_dataset_summary = pd.DataFrame(
    nebius_dataset_rows
)


display(
    nebius_dataset_summary.style.format(
        {
            "Traj.":
                "{:,}",

            "Non-Successful Traj.":
                "{:,}",

            "Unique PRs":
                "{:,}",

            "Successful Attempt PRs":
                "{:,}",

            "Non-Successful Attempt PRs":
                "{:,}",
        }
    )
)

# %%
# %%
assert (
    nebius_dataset_summary[
        "Traj."
    ].sum()
    == 80036
)

assert (
    nebius_dataset_summary[
        "Non-Successful Traj."
    ].sum()
    == 66647
)

print(
    "Nebius total trajectory checks passed."
)


,Dataset,Agent,Model,Traj.,Non-Successful Traj.,Unique PRs,Successful Attempt PRs,Non-Successful Attempt PRs
0,Nebius,SWE,Llama-405B,"1,191",883,100,48,98
1,Nebius,SWE,Llama-70B,"74,792","62,325","3,591",835,"3,555"
2,Nebius,SWE,Llama-8B,"4,053","3,439",528,87,523


Nebius total trajectory checks passed.


In [10]:
# %%
assert (
    nebius_dataset_summary[
        "Traj."
    ].sum()
    == 80036
)

assert (
    nebius_dataset_summary[
        "Non-Successful Traj."
    ].sum()
    == 66647
)

print(
    "Nebius total trajectory checks passed."
)

Nebius total trajectory checks passed.


## Table 1
<a id="table-rq1"></a>

In [9]:
# ============================================================
# Cell 9: Final combined dataset-summary table
# ============================================================

dataset_summary = pd.concat(
    [
        nvidia_dataset_summary,
        nebius_dataset_summary,
    ],
    ignore_index=True,
)


display(
    dataset_summary.style.format(
        {
            "Traj.":
                "{:,}",

            "Non-Successful Traj.":
                "{:,}",

            "Unique PRs":
                "{:,}",

            "Successful Attempt PRs":
                "{:,}",

            "Non-Successful Attempt PRs":
                "{:,}",
        }
    )
)


,Dataset,Agent,Model,Traj.,Non-Successful Traj.,Unique PRs,Successful Attempt PRs,Non-Successful Attempt PRs
0,NVIDIA,OpenHands,MiniMax-M2.5,"49,948","34,007","20,098","6,993","14,885"
1,NVIDIA,OpenHands,Qwen3.5-122B,"55,488","41,831","20,362","6,183","16,752"
2,NVIDIA,SWE,MiniMax-M2.5,"57,268","37,717","20,791","8,106","15,164"
3,NVIDIA,SWE,Qwen3.5-122B,"44,785","28,690","18,211","7,350","12,977"
4,Nebius,SWE,Llama-405B,"1,191",883,100,48,98
5,Nebius,SWE,Llama-70B,"74,792","62,325","3,591",835,"3,555"
6,Nebius,SWE,Llama-8B,"4,053","3,439",528,87,523


In [11]:
# ============================================================
# Cell 10: Load Nebius final TP annotations
# ============================================================

def recover_nebius_instance_id(
    trajectory_id
):

    if trajectory_id is None:
        return None

    trajectory_id = str(
        trajectory_id
    ).strip()

    if not trajectory_id:
        return None


    return re.sub(
        r"_\d+$",
        "",
        trajectory_id,
    )

nebius_annotation_canonical = {}

raw_annotation_rows = 0
duplicate_annotation_rows = 0


with NEBIUS_ANNOTATION_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue


        record = json.loads(
            line
        )


        trajectory_id = record.get(
            "trajectory_id"
        )

        if trajectory_id is None:
            continue


        trajectory_id = str(
            trajectory_id
        ).strip()


        raw_model = str(
            record.get(
                "model_name",
                "",
            )
        ).strip().lower()


        model = (
            NEBIUS_ANNOTATION_MODEL_MAP.get(
                raw_model
            )
        )


        if model is None:

            raise ValueError(
                f"Unknown annotation model "
                f"at line {line_number}: "
                f"{raw_model}"
            )


        key = (
            model,
            trajectory_id,
        )


        raw_annotation_rows += 1


        if key in nebius_annotation_canonical:

            duplicate_annotation_rows += 1


        record = record.copy()

        record[
            "_canonical_model"
        ] = model

        record[
            "_instance_id"
        ] = recover_nebius_instance_id(
            trajectory_id
        )


        # Last occurrence wins
        nebius_annotation_canonical[
            key
        ] = record


print(
    f"Raw annotation rows : "
    f"{raw_annotation_rows:,}"
)

print(
    f"Duplicate rows      : "
    f"{duplicate_annotation_rows:,}"
)

print(
    f"Canonical rows      : "
    f"{len(nebius_annotation_canonical):,}"
)


Raw annotation rows : 9,320
Duplicate rows      : 0
Canonical rows      : 9,320


In [12]:
# ============================================================
# Cell 11: Nebius TP sets
# ============================================================

nebius_tp = {
    model: {
        "tp_trajectories":
            set(),

        "tp_prs":
            set(),
    }

    for model in NEBIUS_MODELS
}


classification_counts = Counter()


for (
    model,
    trajectory_id,
), record in (
    nebius_annotation_canonical.items()
):

    classification = str(
        record.get(
            "classification",
            "",
        )
    ).strip().upper()


    classification_counts[
        classification
    ] += 1


    if classification != "TEST_PANDERING":
        continue


    instance_id = record[
        "_instance_id"
    ]


    if instance_id is None:

        raise ValueError(
            f"Cannot recover PR from "
            f"{trajectory_id}"
        )


    nebius_tp[
        model
    ][
        "tp_trajectories"
    ].add(
        trajectory_id
    )


    nebius_tp[
        model
    ][
        "tp_prs"
    ].add(
        instance_id
    )


print(
    "Classification counts:"
)

print(
    classification_counts
)


Classification counts:
Counter({'TEST_PANDERING': 9320})


In [13]:
nebius_tp_summary_rows = []


for model in NEBIUS_MODELS:

    tp_traj = (
        nebius_tp[
            model
        ][
            "tp_trajectories"
        ]
    )

    tp_prs = (
        nebius_tp[
            model
        ][
            "tp_prs"
        ]
    )


    outside = (
        tp_prs
        -
        nebius_stats[
            model
        ][
            "non_successful_attempt_prs"
        ]
    )


    assert (
        len(outside)
        == 0
    )


    nebius_tp_summary_rows.append(
        {
            "Model":
                model,

            "TP Trajectories":
                len(
                    tp_traj
                ),

            "TP-Affected PRs":
                len(
                    tp_prs
                ),

            "TP Outside NS Population":
                len(
                    outside
                ),
        }
    )


nebius_tp_summary = pd.DataFrame(
    nebius_tp_summary_rows
)


display(
    nebius_tp_summary.style.format(
        {
            "TP Trajectories":
                "{:,}",

            "TP-Affected PRs":
                "{:,}",

            "TP Outside NS Population":
                "{:,}",
        }
    )
)

,Model,TP Trajectories,TP-Affected PRs,TP Outside NS Population
0,Llama-405B,102,45,0
1,Llama-70B,"8,866","2,195",0
2,Llama-8B,352,192,0


In [14]:
# ============================================================
# Cell 12A: NVIDIA full-corpus RQ1 prevalence
# ============================================================

nvidia_full_rows = []


for config in NVIDIA_GROUPS:

    s = nvidia_stats[
        config
    ]

    tp = nvidia_tp[
        config
    ]

    info = NVIDIA_CONFIG_INFO[
        config
    ]


    denominator = len(
        s[
            "non_successful_attempt_prs"
        ]
    )

    numerator = len(
        tp[
            "tp_prs"
        ]
    )

    rate = (
        numerator
        / denominator
        * 100
    )


    nvidia_full_rows.append(
        {
            "Dataset":
                "NVIDIA",

            "Agent":
                info["agent"],

            "Model":
                info["model"],

            "Non-Successful Traj. TP":
                len(
                    tp[
                        "tp_trajectories"
                    ]
                ),

            "Full NS Attempt PRs":
                denominator,

            "Full TP-Affected PRs":
                numerator,

            "Full TP Rate (%)":
                rate,
        }
    )


nvidia_full = pd.DataFrame(
    nvidia_full_rows
)


display(
    nvidia_full.style.format(
        {
            "Non-Successful Traj. TP":
                "{:,}",

            "Full NS Attempt PRs":
                "{:,}",

            "Full TP-Affected PRs":
                "{:,}",

            "Full TP Rate (%)":
                "{:.2f}",
        }
    )
)


,Dataset,Agent,Model,Non-Successful Traj. TP,Full NS Attempt PRs,Full TP-Affected PRs,Full TP Rate (%)
0,NVIDIA,OpenHands,MiniMax-M2.5,"1,059","14,885",950,6.38
1,NVIDIA,OpenHands,Qwen3.5-122B,759,"16,752",707,4.22
2,NVIDIA,SWE,MiniMax-M2.5,936,"15,164",819,5.40
3,NVIDIA,SWE,Qwen3.5-122B,710,"12,977",661,5.09


In [15]:
# ============================================================
# Cell 12B: Nebius full-corpus RQ1 prevalence
# ============================================================

nebius_full_rows = []


for model in NEBIUS_MODELS:

    s = nebius_stats[
        model
    ]

    tp = nebius_tp[
        model
    ]


    denominator = len(
        s[
            "non_successful_attempt_prs"
        ]
    )

    numerator = len(
        tp[
            "tp_prs"
        ]
    )

    rate = (
        numerator
        / denominator
        * 100
    )


    nebius_full_rows.append(
        {
            "Dataset":
                "Nebius",

            "Agent":
                "SWE",

            "Model":
                model,

            "Non-Successful Traj. TP":
                len(
                    tp[
                        "tp_trajectories"
                    ]
                ),

            "Full NS Attempt PRs":
                denominator,

            "Full TP-Affected PRs":
                numerator,

            "Full TP Rate (%)":
                rate,
        }
    )


nebius_full = pd.DataFrame(
    nebius_full_rows
)


display(
    nebius_full.style.format(
        {
            "Non-Successful Traj. TP":
                "{:,}",

            "Full NS Attempt PRs":
                "{:,}",

            "Full TP-Affected PRs":
                "{:,}",

            "Full TP Rate (%)":
                "{:.2f}",
        }
    )
)


,Dataset,Agent,Model,Non-Successful Traj. TP,Full NS Attempt PRs,Full TP-Affected PRs,Full TP Rate (%)
0,Nebius,SWE,Llama-405B,102,98,45,45.92
1,Nebius,SWE,Llama-70B,"8,866","3,555","2,195",61.74
2,Nebius,SWE,Llama-8B,352,523,192,36.71


In [17]:
# ============================================================
# Cell 13: NVIDIA four-configuration common coverage
# ============================================================

nvidia_common_prs = set.intersection(
    *[
        nvidia_stats[
            config
        ][
            "all_prs"
        ]

        for config
        in NVIDIA_GROUPS
    ]
)


print(
    "NVIDIA common PRs across all 4 configs:",
    f"{len(nvidia_common_prs):,}"
)


assert (
    len(nvidia_common_prs)
    == 15605
)
nvidia_common_rows = []


for config in NVIDIA_GROUPS:

    info = NVIDIA_CONFIG_INFO[
        config
    ]


    common_ns_prs = (
        nvidia_common_prs
        &
        nvidia_stats[
            config
        ][
            "non_successful_attempt_prs"
        ]
    )


    common_tp_prs = (
        nvidia_common_prs
        &
        nvidia_tp[
            config
        ][
            "tp_prs"
        ]
    )


    # TP numerator must belong to denominator
    assert (
        common_tp_prs
        <= common_ns_prs
    )


    rate = (
        len(
            common_tp_prs
        )
        /
        len(
            common_ns_prs
        )
        * 100
    )


    nvidia_common_rows.append(
        {
            "Dataset":
                "NVIDIA",

            "Agent":
                info["agent"],

            "Model":
                info["model"],

            "Common PRs":
                len(
                    nvidia_common_prs
                ),

            "Common NS Attempt PRs":
                len(
                    common_ns_prs
                ),

            "Common TP-Affected PRs":
                len(
                    common_tp_prs
                ),

            "Common TP Rate (%)":
                rate,
        }
    )


nvidia_common = pd.DataFrame(
    nvidia_common_rows
)


display(
    nvidia_common.style.format(
        {
            "Common PRs":
                "{:,}",

            "Common NS Attempt PRs":
                "{:,}",

            "Common TP-Affected PRs":
                "{:,}",

            "Common TP Rate (%)":
                "{:.2f}",
        }
    )
)


NVIDIA common PRs across all 4 configs: 15,605


,Dataset,Agent,Model,Common PRs,Common NS Attempt PRs,Common TP-Affected PRs,Common TP Rate (%)
0,NVIDIA,OpenHands,MiniMax-M2.5,"15,605","11,468",699,6.10
1,NVIDIA,OpenHands,Qwen3.5-122B,"15,605","12,612",468,3.71
2,NVIDIA,SWE,MiniMax-M2.5,"15,605","11,134",571,5.13
3,NVIDIA,SWE,Qwen3.5-122B,"15,605","10,899",527,4.84


In [18]:
# ============================================================
# Cell 14: Nebius three-model common coverage
# ============================================================

nebius_common_prs = set.intersection(
    *[
        nebius_stats[
            model
        ][
            "all_prs"
        ]

        for model
        in NEBIUS_MODELS
    ]
)


print(
    "Nebius common PRs across all 3 models:",
    f"{len(nebius_common_prs):,}"
)

nebius_common_rows = []


for model in NEBIUS_MODELS:

    common_ns_prs = (
        nebius_common_prs
        &
        nebius_stats[
            model
        ][
            "non_successful_attempt_prs"
        ]
    )


    common_tp_prs = (
        nebius_common_prs
        &
        nebius_tp[
            model
        ][
            "tp_prs"
        ]
    )


    assert (
        common_tp_prs
        <= common_ns_prs
    )


    rate = (
        len(
            common_tp_prs
        )
        /
        len(
            common_ns_prs
        )
        * 100
    )


    nebius_common_rows.append(
        {
            "Dataset":
                "Nebius",

            "Agent":
                "SWE",

            "Model":
                model,

            "Common PRs":
                len(
                    nebius_common_prs
                ),

            "Common NS Attempt PRs":
                len(
                    common_ns_prs
                ),

            "Common TP-Affected PRs":
                len(
                    common_tp_prs
                ),

            "Common TP Rate (%)":
                rate,
        }
    )


nebius_common = pd.DataFrame(
    nebius_common_rows
)


display(
    nebius_common.style.format(
        {
            "Common PRs":
                "{:,}",

            "Common NS Attempt PRs":
                "{:,}",

            "Common TP-Affected PRs":
                "{:,}",

            "Common TP Rate (%)":
                "{:.2f}",
        }
    )
)

Nebius common PRs across all 3 models: 50


,Dataset,Agent,Model,Common PRs,Common NS Attempt PRs,Common TP-Affected PRs,Common TP Rate (%)
0,Nebius,SWE,Llama-405B,50,48,25,52.08
1,Nebius,SWE,Llama-70B,50,47,37,78.72
2,Nebius,SWE,Llama-8B,50,48,35,72.92


## Table 2
<a id="table-rq2-1"></a>


In [19]:
# ============================================================
# Cell 15: Final RQ1 prevalence table
# ============================================================

full_table = pd.concat(
    [
        nvidia_full,
        nebius_full,
    ],
    ignore_index=True,
)


common_table = pd.concat(
    [
        nvidia_common,
        nebius_common,
    ],
    ignore_index=True,
)


rq1_prevalence = (
    full_table.merge(
        common_table[
            [
                "Dataset",
                "Agent",
                "Model",
                "Common NS Attempt PRs",
                "Common TP-Affected PRs",
                "Common TP Rate (%)",
            ]
        ],
        on=[
            "Dataset",
            "Agent",
            "Model",
        ],
        how="left",
        validate="one_to_one",
    )
)


display(
    rq1_prevalence.style.format(
        {
            "Non-Successful Traj. TP":
                "{:,}",

            "Full NS Attempt PRs":
                "{:,}",

            "Full TP-Affected PRs":
                "{:,}",

            "Full TP Rate (%)":
                "{:.2f}",

            "Common NS Attempt PRs":
                "{:,}",

            "Common TP-Affected PRs":
                "{:,}",

            "Common TP Rate (%)":
                "{:.2f}",
        }
    )
)

,Dataset,Agent,Model,Non-Successful Traj. TP,Full NS Attempt PRs,Full TP-Affected PRs,Full TP Rate (%),Common NS Attempt PRs,Common TP-Affected PRs,Common TP Rate (%)
0,NVIDIA,OpenHands,MiniMax-M2.5,"1,059","14,885",950,6.38,"11,468",699,6.10
1,NVIDIA,OpenHands,Qwen3.5-122B,759,"16,752",707,4.22,"12,612",468,3.71
2,NVIDIA,SWE,MiniMax-M2.5,936,"15,164",819,5.40,"11,134",571,5.13
3,NVIDIA,SWE,Qwen3.5-122B,710,"12,977",661,5.09,"10,899",527,4.84
4,Nebius,SWE,Llama-405B,102,98,45,45.92,48,25,52.08
5,Nebius,SWE,Llama-70B,"8,866","3,555","2,195",61.74,47,37,78.72
6,Nebius,SWE,Llama-8B,352,523,192,36.71,48,35,72.92
